In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import nibabel as nib
from skimage.transform import resize
from tqdm import tqdm
import matplotlib.pyplot as plt

In [3]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# Parametri base
base_dir = "Dataset"
image_types = ["Flair", "T1", "T2"]
target_type = "LesionSeg-Flair"
img_size = (64, 64)  # 2D

In [5]:
from skimage.transform import resize
import numpy as np
import random

class BrainMRISliceDataset(Dataset):
    def __init__(self, image_data, image_types, target_type, img_size=(128, 128), depth=64, augment=False):
        self.slices = []
        self.augment = augment
        H, W = img_size

        for pid, data in image_data.items():
            # Ridimensiono le 3 modalità con anti_aliasing (ok per immagini)
            imgs = [resize(data[t], (H, W, depth), anti_aliasing=True, preserve_range=True) for t in image_types]

            # Ridimensiono la mask SENZA anti_aliasing (evita valori intermedi) e poi binarizzo
            target = resize(data[target_type], (H, W, depth), anti_aliasing=False, preserve_range=True)
            target = (target > 0.5).astype(np.float32)

            for z in range(depth):
                input_slice = np.stack([img[..., z] for img in imgs]).astype(np.float32)  # (C,H,W)
                target_slice = target[..., z].astype(np.float32)  # (H,W)

                # tieni solo slice con lesione > 0
                if np.any(target_slice > 0):
                    self.slices.append((input_slice, target_slice))

    def __len__(self):
        return len(self.slices)

    def _augment(self, x, y):
        # x: (C,H,W), y: (H,W)
        # flip orizz/vert
        if random.random() < 0.5:
            x = np.flip(x, axis=2).copy()
            y = np.flip(y, axis=1).copy()
        if random.random() < 0.5:
            x = np.flip(x, axis=1).copy()
            y = np.flip(y, axis=0).copy()
        # rotazione 0/90/180/270 coerente
        k = random.randint(0, 3)
        if k:
            x = np.rot90(x, k, axes=(1, 2)).copy()
            y = np.rot90(y, k, axes=(0, 1)).copy()
        return x, y

    def __getitem__(self, idx):
        x, y = self.slices[idx]
        if self.augment:
            x, y = self._augment(x, y)
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32).unsqueeze(0)  # (1,H,W)
        return x, y



In [ ]:
# Caricamento immagini
image_data = {}
for i in tqdm(range(1, 61), desc="Caricamento immagini"):
    #merge tra tabelle
    folder_path = os.path.join(base_dir, f"Patient-{i}")
    patient_images = {}
    for t in image_types + [target_type]:
        #merge tra tabelle
        file_path = os.path.join(folder_path, f"{i}-{t}.nii")
        if os.path.exists(file_path):
            img = nib.load(file_path).get_fdata()
            img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
            patient_images[t] = img
    if len(patient_images) == len(image_types) + 1:
        image_data[i] = patient_images

# Split e loader
#suddividione train/test
train_ids, val_ids = train_test_split(list(image_data.keys()), test_size=0.2, random_state=42)
train_data = {i: image_data[i] for i in train_ids}
val_data = {i: image_data[i] for i in val_ids}

train_dataset = BrainMRISliceDataset(train_data, image_types, target_type)
val_dataset = BrainMRISliceDataset(val_data, image_types, target_type)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

Caricamento immagini: 100%|██████████| 60/60 [00:18<00:00,  3.32it/s]


In [7]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)

class UNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base=32):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_ch, base)          # 32
        self.enc2 = DoubleConv(base, base*2)         # 64
        self.enc3 = DoubleConv(base*2, base*4)       # 128
        self.enc4 = DoubleConv(base*4, base*8)       # 256
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(base*8, base*16) # 512

        # Decoder (upconv + double conv)
        self.up4 = nn.ConvTranspose2d(base*16, base*8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base*16, base*8)

        self.up3 = nn.ConvTranspose2d(base*8, base*4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)

        self.up2 = nn.ConvTranspose2d(base*4, base*2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)

        self.up1 = nn.ConvTranspose2d(base*2, base, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base*2, base)

        self.outc = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b  = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = torch.cat([d4, e4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return self.outc(d1)


In [8]:
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
    def forward(self, inputs, targets):
        probs = torch.sigmoid(inputs)
        # calcolo per batch
        dims = (1,2,3)
        inter = (probs * targets).sum(dims)
        den   = probs.sum(dims) + targets.sum(dims) + self.eps
        dice  = (2. * inter) / den
        return 1 - dice.mean()

def combo_loss(logits, targets, bce_w=0.5):
    bce  = F.binary_cross_entropy_with_logits(logits, targets)
    dsc  = DiceLoss()(logits, targets)
    return bce_w*bce + (1-bce_w)*dsc


In [9]:
def soft_dice(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1,2,3)
    inter = (probs * targets).sum(dims)
    den   = probs.sum(dims) + targets.sum(dims) + eps
    dice  = (2. * inter) / den
    return dice.mean().item()


In [ ]:
#Calcolo delle metriche di valutazione
def soft_dice(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    dims = (1, 2, 3)
    inter = (probs * targets).sum(dims)
    den   = probs.sum(dims) + targets.sum(dims) + eps
    dice  = (2.0 * inter) / den
    return dice.mean().item()

def precision_score(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    TP = (preds * targets).sum()
    FP = (preds * (1 - targets)).sum()
    return (TP / (TP + FP + eps)).item()

def recall_score(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    TP = (preds * targets).sum()
    FN = ((1 - preds) * targets).sum()
    return (TP / (TP + FN + eps)).item()


# Modello + ottimizzazione 
model = UNet2D(in_ch=3, out_ch=1, base=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

use_cuda_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler('cuda', enabled=use_cuda_amp)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5
)

EPOCHS = 80
best_dice = 0.0

for epoch in range(1, EPOCHS + 1):
    # -------------------
    # TRAIN
    # -------------------
    model.train()
    train_loss = 0.0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=use_cuda_amp):
            out = model(x)
            loss = combo_loss(out, y, bce_w=0.5)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

    train_loss /= max(1, len(train_loader))

    # -------------------
    # VALIDATION
    # -------------------
    model.eval()
    val_dice = 0.0
    val_prec = 0.0   
    val_rec  = 0.0   

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)

            val_dice += soft_dice(out, y)
            #calcolo della precision
            val_prec += precision_score(out, y)
            #calcolo della recall
            val_rec  += recall_score(out, y)     

    # medie sulle batch
    n_val = max(1, len(val_loader))
    val_dice /= n_val
    val_prec /= n_val
    val_rec  /= n_val

    # step dello scheduler sulla metrica principale (Dice)
    scheduler.step(val_dice)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val SoftDice: {val_dice:.4f} | "
        f"Precision: {val_prec:.4f} | "
        f"Recall: {val_rec:.4f}"
    )

    # checkpoint sul migliore secondo il Dice
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "unet2d_best.pth")


Epoch 1/80 | Loss: 0.5625 | Val SoftDice: 0.0396 | Precision: 0.1529 | Recall: 0.2822
Epoch 2/80 | Loss: 0.4591 | Val SoftDice: 0.1792 | Precision: 0.1826 | Recall: 0.6706
Epoch 3/80 | Loss: 0.3749 | Val SoftDice: 0.2345 | Precision: 0.6588 | Recall: 0.1785
Epoch 4/80 | Loss: 0.3339 | Val SoftDice: 0.3583 | Precision: 0.5466 | Recall: 0.3648
Epoch 5/80 | Loss: 0.3209 | Val SoftDice: 0.3876 | Precision: 0.5797 | Recall: 0.3958
Epoch 6/80 | Loss: 0.3153 | Val SoftDice: 0.3388 | Precision: 0.3526 | Recall: 0.5221
Epoch 7/80 | Loss: 0.3052 | Val SoftDice: 0.4181 | Precision: 0.5099 | Recall: 0.4810
Epoch 8/80 | Loss: 0.3006 | Val SoftDice: 0.4115 | Precision: 0.5108 | Recall: 0.4589
Epoch 9/80 | Loss: 0.2943 | Val SoftDice: 0.3418 | Precision: 0.6288 | Recall: 0.3097
Epoch 10/80 | Loss: 0.2909 | Val SoftDice: 0.4373 | Precision: 0.5323 | Recall: 0.4955
Epoch 11/80 | Loss: 0.2899 | Val SoftDice: 0.4392 | Precision: 0.5421 | Recall: 0.4703
Epoch 12/80 | Loss: 0.2819 | Val SoftDice: 0.4150 | 